ERA5 reanalysis atmospheres via a tabulated density profile
===========================================================

MCEq can run on a **measured or reanalysed atmosphere** instead of a
parametrization, through
`MCEq.geometry.density_profiles.TabulatedAtmosphere`. The class reads a small
CSV table; converting a dataset into that table is left to the user, because
every archive has its own API and file format and MCEq does not want to depend
on any of them.

This notebook is the worked example for one such archive: **ERA5**, the ECMWF
reanalysis. It

1. downloads one ERA5 pressure-level field from the Copernicus Climate Data
   Store (CDS),
2. extracts the vertical profile above a site and writes the MCEq table,
3. runs MCEq on it and compares against the NRLMSISE-00 atmosphere for the same
   site and season.

> **This notebook is not executed when the documentation is built.** It needs a
> CDS account and GRIB tooling that MCEq does not depend on, and it downloads a
> few hundred MB. Run it locally.

What you need
-------------

```bash
pip install cdsapi pygrib      # pygrib ships the eccodes C library as a wheel
```

and a CDS account with an API key in `~/.cdsapirc`:

* register at <https://cds.climate.copernicus.eu>
* follow <https://cds.climate.copernicus.eu/how-to-api>
* accept the licence for the *ERA5 hourly data on pressure levels* dataset once,
  from its download page, or the request will be rejected.

The table format
----------------

A table is a CSV file describing **one vertical column, above one site, at one
point in time**:

```
# MCEq tabulated atmosphere v1
# South Pole, ERA5 pressure levels, 2012-01-03 12:00 UTC
h_cm,T_K,p_hPa
283400.0,247.1,681.2
510000.0,231.4,500.0
900000.0,,300.0
```

* `h_cm` is required: height above sea level in cm.
* Density comes either from a `rho_gcm3` column directly, or from `T_K` and
  `p_hPa` via the dry-air ideal gas law.
* `T_K` also feeds `get_temperature()`.
* Comment lines start with `#`, column order does not matter, unknown columns
  are ignored, rows may be in any order, and missing values are written as an
  empty field or `nan` and dropped row-wise.

Anything that can be written in this shape works — AIRS, GDAS, radiosondes, the
output of a regional model. Only the converter below is ERA5-specific.

In [ ]:
import os

import numpy as np

# --- the site and the date to extract -------------------------------------
SITE_NAME = "SouthPole"
SITE_LAT = -90.0  # degrees North
SITE_LON = 0.0  # degrees East
SITE_ELEVATION_M = 2835.0  # surface elevation; ERA5 extrapolates below ground
DATE = "2012-01-03"
TIME = "12:00"
SEASON = "January"  # for the MSIS comparison below

# ERA5 is global; a small box around the site keeps the download to a few MB.
AREA = [SITE_LAT + 2, SITE_LON - 2, SITE_LAT - 2, SITE_LON + 2]  # N, W, S, E

DOWNLOAD_DIR = "era5_data"
GRIB_FILE = os.path.join(DOWNLOAD_DIR, f"era5_{SITE_NAME}_{DATE}_{TIME[:2]}.grib")
TABLE_FILE = f"{SITE_NAME}_{DATE}.csv"

### 1. Download

One request, two variables (geopotential and temperature) on all 37 standard
pressure levels. The file is kept on disk, so re-running the notebook does not
re-download it.

In [ ]:
PRESSURE_LEVELS = [
    "1", "2", "3", "5", "7", "10", "20", "30", "50", "70",
    "100", "125", "150", "175", "200", "225", "250", "300", "350", "400",
    "450", "500", "550", "600", "650", "700", "750", "775", "800", "825",
    "850", "875", "900", "925", "950", "975", "1000",
]

if not os.path.exists(GRIB_FILE):
    import cdsapi

    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    year, month, day = DATE.split("-")
    cdsapi.Client().retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": ["reanalysis"],
            "data_format": "grib",
            "variable": ["geopotential", "temperature"],
            "pressure_level": PRESSURE_LEVELS,
            "year": [year],
            "month": [month],
            "day": [day],
            "time": [TIME],
            "area": AREA,
        },
        GRIB_FILE,
    )
print(f"{GRIB_FILE}: {os.path.getsize(GRIB_FILE) / 1e6:.1f} MB")

### 2. GRIB → MCEq table

ERA5 pressure-level fields give geopotential `z` (m²/s²) and temperature `t` (K)
on a latitude/longitude grid, one GRIB message per (variable, level). For a
single column we take the grid point nearest the site and read one value per
level.

Two conversions matter:

* **geopotential → height**: `h = z / g0` with the standard gravity
  `g0 = 9.80665 m/s²`. This is *geopotential* height; the difference to
  geometric height is below 1% at 50 km and is ignored here.
* **below-ground levels**: ERA5 extrapolates pressure levels underneath the
  terrain, so at an elevated site the 1000-850 hPa levels are fictitious. They
  are dropped against the known surface elevation.

In [ ]:
G0 = 9.80665  # standard gravity, m/s^2


def era5_profile(grib_file, site_lat, site_lon):
    """Vertical (height, temperature, pressure) profile at the nearest grid point."""
    import pygrib

    columns = {}
    with pygrib.open(grib_file) as grbs:
        for grb in grbs:
            if grb.shortName not in ("z", "t"):
                continue
            lats, lons = grb.latlons()
            # Nearest grid point; longitudes are wrapped into (-180, 180].
            j = np.abs(lats[:, 0] - site_lat).argmin()
            dlon = (lons[0, :] - site_lon + 180.0) % 360.0 - 180.0
            i = np.abs(dlon).argmin()
            columns.setdefault(grb.shortName, {})[grb.level] = float(grb.values[j, i])

    levels = np.array(sorted(set(columns["z"]) & set(columns["t"])), dtype=float)
    h_cm = np.array([columns["z"][int(p)] for p in levels]) / G0 * 1e2
    t_k = np.array([columns["t"][int(p)] for p in levels])

    order = np.argsort(h_cm)
    return h_cm[order], t_k[order], levels[order]


def write_table(filename, h_cm, t_k, p_hpa, header_comment):
    """Writes a v1 MCEq tabulated-atmosphere CSV."""
    with open(filename, "w") as out:
        out.write("# MCEq tabulated atmosphere v1\n")
        out.write(f"# {header_comment}\n")
        out.write("h_cm,T_K,p_hPa\n")
        for h, t, p in zip(h_cm, t_k, p_hpa):
            out.write(f"{h:.6e},{t:.4f},{p:.6g}\n")


h_cm, t_k, p_hpa = era5_profile(GRIB_FILE, SITE_LAT, SITE_LON)

# Drop the levels that sit below the terrain.
above_ground = h_cm >= SITE_ELEVATION_M * 1e2
h_cm, t_k, p_hpa = h_cm[above_ground], t_k[above_ground], p_hpa[above_ground]

write_table(
    TABLE_FILE,
    h_cm,
    t_k,
    p_hpa,
    f"{SITE_NAME} ({SITE_LAT:.2f} N, {SITE_LON:.2f} E), "
    f"ERA5 pressure levels, {DATE} {TIME} UTC",
)
print(f"{TABLE_FILE}: {len(h_cm)} levels, "
      f"{h_cm[0] / 1e5:.2f} - {h_cm[-1] / 1e5:.2f} km")
print(open(TABLE_FILE).read(300))

> **Model levels reach higher.** The 37 pressure levels stop at 1 hPa, about
> 48 km. ERA5's 137 *model* levels reach ~80 km, at the cost of a
> `reanalysis-era5-complete` (MARS) request and a hydrostatic integration of the
> hybrid sigma-pressure coordinate. That path is not shown here; the recipe is
> to read the half-level coefficients `a`, `b` from the GRIB `pv` array, form
> `p_{k+1/2} = a + b * exp(lnsp)`, and integrate
> `dPhi = R_d * T_v * ln(p_low / p_up)` upward from the surface geopotential.
>
> For the flux it matters little: above 48 km only ~1 g/cm² of column is left,
> ~0.1% of a vertical atmosphere, and `TabulatedAtmosphere` covers it with an
> isothermal tail. Pass `top_extension="msis00"` to blend into NRLMSISE-00 there
> instead.

### 3. Run MCEq on the table

`("Tabulated", (path,))` is all `set_density_model` needs. By default the
observation level is taken from the lowest row of the table — here the ice
surface at 2835 m — which is also where `MSIS00_IC` ends its column, so the two
models are directly comparable.

In [ ]:
import crflux.models as pm
import matplotlib.pyplot as plt

import MCEq.geometry.density_profiles as dp
from MCEq.core import MCEqRun

era5_atm = dp.TabulatedAtmosphere(
    TABLE_FILE, location=SITE_NAME, season=SEASON
)
msis_atm = dp.MSIS00IceCubeCentered(SITE_NAME, SEASON)

print(f"ERA5 observation level {era5_atm.geom.h_obs / 1e5:.3f} km, "
      f"max zenith {era5_atm.max_theta:.2f} deg")
print(f"MSIS observation level {msis_atm.geom.h_obs / 1e5:.3f} km")

In [ ]:
heights = np.linspace(era5_atm.geom.h_obs, era5_atm.geom.h_atm, 400)


def profile(atm, getter_name, heights):
    """Sample an atmosphere on a height grid.

    Looped rather than vectorised because the NRLMSISE-00 wrapper is a scalar
    ctypes call; the tabulated model would take the array directly.
    """
    getter = getattr(atm, getter_name)
    return np.array([float(getter(h)) for h in heights])


era5_rho = profile(era5_atm, "get_density", heights)
msis_rho = profile(msis_atm, "get_density", heights)
era5_t = profile(era5_atm, "get_temperature", heights)
msis_t = profile(msis_atm, "get_temperature", heights)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=120)

axes[0].semilogx(era5_rho, heights / 1e5, label="ERA5")
axes[0].semilogx(msis_rho, heights / 1e5, "--", label="MSIS00")
axes[0].axhline(h_cm[-1] / 1e5, color="0.6", lw=0.8, ls=":")
axes[0].set_xlabel(r"$\rho$ [g/cm$^3$]")
axes[0].set_ylabel("height [km]")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(era5_t, heights / 1e5, label="ERA5")
axes[1].plot(msis_t, heights / 1e5, "--", label="MSIS00")
axes[1].axhline(h_cm[-1] / 1e5, color="0.6", lw=0.8, ls=":")
axes[1].set_xlabel("T [K]")
axes[1].grid(alpha=0.3)

axes[2].plot(era5_rho / msis_rho, heights / 1e5)
axes[2].axvline(1.0, color="0.6", lw=0.8)
axes[2].axhline(h_cm[-1] / 1e5, color="0.6", lw=0.8, ls=":")
axes[2].set_xlabel(r"$\rho_{\rm ERA5} / \rho_{\rm MSIS00}$")
axes[2].set_xlim(0.5, 1.5)
axes[2].grid(alpha=0.3)

fig.suptitle(
    f"{SITE_NAME}, {DATE} — dotted line: top of the ERA5 table, "
    "isothermal tail above"
)
fig.tight_layout()

### 4. Slant depth and lepton flux

The zenith range stops at the horizon: the table is a single column above the
site, so it says nothing about the atmosphere at the impact point of an inclined
shower, and nothing at all about the far side of the Earth. For upgoing or
azimuth-dependent atmospheres use `MSIS00LocationCentered` /
`MSIS21LocationCentered`, which resolve the impact point.

In [ ]:
ZENITHS = [0.0, 30.0, 60.0, 85.0]

mceq_era5 = MCEqRun(
    interaction_model="SIBYLL2.3d",
    primary_model=(pm.HillasGaisser2012, "H3a"),
    theta_deg=0.0,
    density_model=era5_atm,
)
mceq_msis = MCEqRun(
    interaction_model="SIBYLL2.3d",
    primary_model=(pm.HillasGaisser2012, "H3a"),
    theta_deg=0.0,
    density_model=msis_atm,
)

flux = {}
for label, mceq in (("ERA5", mceq_era5), ("MSIS00", mceq_msis)):
    for zenith in ZENITHS:
        mceq.set_zenith_azimuth(zenith)
        mceq.solve()
        flux[label, zenith] = mceq.get_solution("conv_numu") + mceq.get_solution(
            "conv_antinumu"
        )
        print(f"{label:7s} zenith {zenith:5.1f} deg  "
              f"max_X = {mceq.density_model.max_X:8.2f} g/cm^2")

e_grid = mceq_era5.e_grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)

for zenith in ZENITHS:
    line, = axes[0].loglog(
        e_grid, e_grid**3 * flux["ERA5", zenith], label=f"{zenith:.0f}$^\\circ$"
    )
    axes[0].loglog(
        e_grid, e_grid**3 * flux["MSIS00", zenith], "--", color=line.get_color()
    )
    axes[1].semilogx(
        e_grid, flux["ERA5", zenith] / flux["MSIS00", zenith], color=line.get_color()
    )

axes[0].set_xlabel("E [GeV]")
axes[0].set_ylabel(r"$E^3 \Phi_{\nu_\mu + \bar\nu_\mu}$ [GeV$^2$ cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
axes[0].set_title("solid: ERA5, dashed: MSIS00")
axes[0].legend(title="zenith")
axes[0].grid(alpha=0.3)

axes[1].axhline(1.0, color="0.6", lw=0.8)
axes[1].set_xlabel("E [GeV]")
axes[1].set_ylabel("ERA5 / MSIS00")
axes[1].set_title("conventional $\\nu_\\mu + \\bar\\nu_\\mu$")
axes[1].grid(alpha=0.3)

fig.tight_layout()

### Where to go from here

* **Other datasets.** Only the converter in section 2 is ERA5-specific. Write
  `h_cm` plus `T_K`/`p_hPa` (or `rho_gcm3`) and any archive works.
* **Seasonal studies.** One table per date; build one `TabulatedAtmosphere` per
  table and hand them to `MCEqRun.solve_batch(..., conditions=...)` as
  per-member `density_model`s.
* **Direction-resolved atmospheres.** A table holds one column, so the shower
  impact point is not resolved and the azimuth angle is ignored. Extending the
  format with `lat_deg`/`lon_deg` columns, so the profile can be sampled where
  the shower actually develops (and upgoing angles supported), is tracked on the
  MCEq issue tracker.